# 1 pamoka – Įvadas į DI agentus

Sveiki atvykę į pirmąją **DI agentų pradedantiesiems** kurso pamoką!

**DI agentas** yra programa, kuri naudoja didelio masto kalbos modelį (LLM) kaip savo sprendimų variklį ir gali atlikti *veiksmus* realiame pasaulyje — kviesti API, užklausti duomenų bazes arba vykdyti kodą — siekdama įgyvendinti naudotojo užduotį.

Šiame užrašų knygelėje sukursite savo pirmąjį agentą: **Kelionių agentą**, kuris rekomenduoja atostogų vietas. Kelyje sužinosite, kaip:

1. Prisijungti prie Microsoft Foundry Agent Service naudodami **Microsoft Agent Framework**.
2. Agentui suteikti **įrankį** — paprastą Python funkciją, kurią jis gali iškviesti.
3. Paleisti agentą ir patikrinti jo atsakymą.
4. Srautiniu būdu gauti agente atsako žodžius po vieną.


## Paruošimas

Prieš paleisdami šį užrašų knygelę, įsitikinkite, kad turite:

1. **Microsoft Foundry projektą** su įdiegta pokalbių modeliu (pvz., `gpt-4.1-mini`).
2. **Prisijungę prie Azure CLI** — terminale paleiskite `az login`.
3. **Nustatytus reikiamus aplinkos kintamuosius:**
   - `AZURE_AI_PROJECT_ENDPOINT` — jūsų Microsoft Foundry projekto galinis taškas.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — jūsų įdiegto modelio pavadinimas.

Žemiau esanti langelis įdiegia jums reikalingas Python paketas.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Sukurkite savo pirmąjį agentą

Agentui reikia dviejų dalykų:

- **Nurodymai**, kurie pasako *kas jis yra* ir *kaip elgtis* (sistemos paskatinimas).
- **Įrankiai** — Python funkcijos, pažymėtos `@tool`, kurias agentas gali iškviesti norėdamas gauti informaciją arba atlikti veiksmus.

Žemiau apibrėžiame paprastą įrankį, kuris grąžina populiarių atostogų vietovių sąrašą. Agentas naudos šį įrankį, kai vartotojas paprašys kelionių rekomendacijų.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Srautinės Atsakymų Teikimas

Norėdami labiau interaktyvios patirties, galite **srautiniu būdu** gauti agento atsakymą. Vietoje to, kad lauktumėte viso atsakymo pabaigos, agentas pateikia teksto dalis, kai jos sugeneruojamos. Tai ypač naudinga pokalbių sąsajose, kur norite realiu laiku rodyti rezultatus.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Santrauka

Šioje pamokoje sužinojote, kaip:

- **Sukurti tiekėją**, kuris jungiasi prie Microsoft Foundry Agent Service per `FoundryChatClient`.
- **Apibrėžti įrankį** naudojant `@tool` dekoratorių, kad agentas galėtų kviesti jūsų Python funkcijas.
- **Paleisti agentą** su vartotojo žinute ir atspausdinti jo atsakymą.
- **Srautiniu būdu perduoti atsakymus** realaus laiko išvesties rodymui.

Kitoje pamokoje gilinsimės į agentų sistemas ir išmoksite suteikti agentams galingesnius įrankius bei daugiapakopes samprotavimo galimybes.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojama naudoti profesionalų žmogiškąjį vertimą. Mes neatsakome už jokius nesusipratimus ar neteisingą interpretaciją, kilusią naudojantis šiuo vertimu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
